In [1]:
import math
from itertools import combinations

The baseline probability is $(1 - (255/256)^p)^8$

In [2]:
def P(p):
    alpha = (255/256)**p
    return (1 - alpha)**8

In [3]:
def get_cluster_sizes(L):
    """
    Given a sorted list of indices L (subset of {1..7}), 
    returns a list of cluster sizes.
    Example: L=[1, 2, 4, 5, 6] -> [2, 3] (Cluster {1,2} size 2, Cluster {4,5,6} size 3)
    """
    if not L:
        return []
    
    clusters = []
    current_size = 1
    
    for i in range(1, len(L)):
        if L[i] == L[i-1] + 1:
            current_size += 1
        else:
            clusters.append(current_size)
            current_size = 1
    clusters.append(current_size)
    return clusters


$u_L=1+\sum\limits_{\varnothing\neq K\subset L} (-1)^K \dfrac{1}{2^{|\cup_{j\in K} R_j|}} = \sum\limits_{K\subset L} (-1)^K \dfrac{1}{2^{|\cup_{j\in K} R_j|}}$

$P_{\text{super}} \le 1 + \sum\limits_{\varnothing\neq \sub \{1,\ldots,7\}}(-1)^{|L|}u_L^p$

In [4]:
def uL(L, s):
    n = len(L)
    u_val = 0.0
    
    for r in range(1, n + 1):
        for K in combinations(L, r):
            # Get clusters of K
            k_clusters = get_cluster_sizes(K)
            
            # Total covered bits: Sum(8*size + s - 8) for each cluster
            total_bits = 0
            for size in k_clusters:
                total_bits += (8 * size + (s - 8))
            
            term = math.pow(0.5, total_bits)
            
            if r % 2 == 1:
                u_val += term
            else:       
                u_val -= term
                
    return u_val

def P_super(p, s):
    indices = range(1, 8)
    total_prob = 1.0
    
    # Iterate over all non-empty subsets L of {1..7}
    for r in range(1, 8):
        for L in combinations(indices, r):            
            u_L = uL(L, s)

            term_val = math.pow(1.0 - u_L, p)
            
            if r % 2 == 1:
                total_prob -= term_val
            else:        
                total_prob += term_val
                
    return total_prob

In [9]:
# Test
p = 5000
s_values = [8, 9, 10, 11, 12, 13, 14, 15]

print(f"Upper Bound for P(B) with p={p}:")
print("-" * 25)

for s in s_values:
    prob = P_super(p, s)
    # Clamp to [0, 1] for sanity, though formula naturally stays within bounds usually
    prob = max(0.0, min(1.0, prob)) 
    print(f"{s:<5} | {prob:.10f}")

Upper Bound for P(B) with p=5000:
-------------------------
8     | 0.9999999778
9     | 0.9996021500
10    | 0.9482858619
11    | 0.5291011849
12    | 0.0869876789
13    | 0.0042362160
14    | 0.0000919863
15    | 0.0000012934


In [8]:
def compare(p, s):
    pb = P(p)
    ps = P_super(p, s)

    print(f"p = {p}, s = {s}")
    print(f"P_base  = {pb:.6e}")
    print(f"P_super = {ps:.6e}")
    print(f"ratio (super/base) = {ps/pb:.6e}")
    print()

for p in [10, 50, 100, 200, 400, 1000, 2000]:
    compare(p,12)

p = 10, s = 12
P_base  = 4.710940e-12
P_super = -9.214851e-15
ratio (super/base) = -1.956054e-03

p = 50, s = 12
P_base  = 9.959155e-07
P_super = 1.860734e-13
ratio (super/base) = 1.868365e-07

p = 100, s = 12
P_base  = 1.210907e-04
P_super = 1.075973e-11
ratio (super/base) = 8.885672e-08

p = 200, s = 12
P_base  = 7.543007e-03
P_super = 8.446508e-10
ratio (super/base) = 1.119780e-07

p = 400, s = 12
P_base  = 1.532990e-01
P_super = 7.379347e-08
ratio (super/base) = 4.813697e-07

p = 1000, s = 12
P_base  = 8.510234e-01
P_super = 2.401722e-05
ratio (super/base) = 2.822157e-05

p = 2000, s = 12
P_base  = 9.968164e-01
P_super = 1.320217e-03
ratio (super/base) = 1.324434e-03

